## Mounting Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Full fine-tuning: decoder

Starting from the Phase-1 decoder checkpoint and training every weight
on fold1. fold2 is kept aside to check how well it does on unseen documents.


In [2]:
%cd /content/drive/MyDrive/CS781/phase2
!python train.py --arch decoder \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p2 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/decoder_full


/content/drive/MyDrive/CS781/phase2
Using device: cuda
Number of labels: 25
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
524 train chunks, 527 dev chunks
131 steps per epoch, 393 steps total
step 10/393  epoch 1  loss 3.7386  lr 6.00e-06  elapsed 0.1 min
step 20/393  epoch 1  loss 2.4932  lr 1.20e-05  elapsed 0.2 min
step 30/393  epoch 1  loss 0.7317  lr 1.80e-05  elapsed 0.2 min
step 40/393  epoch 1  loss 0.5103  lr 2.40e-05  elapsed 0.3 min
step 50/393  epoch 1  loss 0.2664  lr 3.00e-05  elapsed 0.3 min
  validation loss: 0.2694
step 60/393  epoch 1  loss 0.2605  lr 3.00e-05  elapsed 0.7 min
step 70/393  epoch 1  loss 0.1569  lr 2.98e-05  elapsed 0.8 min
step 80/393  epoch 1  loss 0.1778  lr 2.95e-05  elapsed 0.8 min
step 90/393  epoch 1  loss 0.1401  lr 2.91e-05  elapsed 0.9 min
step 100/393  epoch 1  loss 0.1226  lr 2.87e-05  elapsed 1.0 min
  validation loss: 0.1789
step 110/393  epoch 

## Copying fold2 text and gold to Drive

Downloading fold2's raw text and gold spans into data_p2, so predict.py
can score its predictions on fold2 before making a Codabench submission.


In [3]:
from huggingface_hub import login, hf_hub_download
import shutil

login()

# Saving both fold2 files next to the other data_p2 files
for name in ["fold2-text.json", "fold2-gold.json"]:
    path = hf_hub_download(
        "Exploration-Lab/CS-781-Capstone",
        f"Phase_2_LNER/Train/{name}",
        repo_type="dataset"
    )
    shutil.copy(path, f"/content/drive/MyDrive/CS781/data_p2/{name}")


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

## Scoring decoder_full on fold2

Running predict.py on fold2 with strict matching, as a check that the
whole pipeline works before submitting on fold3. Using merge decoding,
so pieces of one word are not split into separate entities.



In [5]:
!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/decoder_full/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/fold2-text.json \
    --gold_file /content/drive/MyDrive/CS781/data_p2/fold2-gold.json \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/decoder_full/dev_check \
    --decoding merge


Loaded decoder NER checkpoint from step 250, val loss 0.1496
Decoding: merge
tagged 5/35 documents
tagged 10/35 documents
tagged 15/35 documents
tagged 20/35 documents
tagged 25/35 documents
tagged 30/35 documents
tagged 35/35 documents

Wrote /content/drive/MyDrive/CS781/ckpt_p2/decoder_full/dev_check/prediction.json
4896 predicted spans across 35 documents

Strict macro P 0.3418  R 0.3655  F1 0.3419

  APP         P 0.043  R 0.040  F1 0.042  (gold 200, pred 185)
  RESP        P 0.213  R 0.084  F1 0.121  (gold 190, pred 75)
  A.COUNSEL   P 0.347  R 0.464  F1 0.397  (gold 125, pred 167)
  R.COUNSEL   P 0.352  R 0.330  F1 0.340  (gold 97, pred 91)
  JUDGE       P 0.299  R 0.459  F1 0.362  (gold 122, pred 187)
  WIT         P 0.219  R 0.058  F1 0.092  (gold 120, pred 32)
  AUTH        P 0.168  R 0.132  F1 0.148  (gold 468, pred 369)
  COURT       P 0.764  R 0.785  F1 0.775  (gold 558, pred 573)
  STAT        P 0.491  R 0.531  F1 0.510  (gold 735, pred 795)
  PREC        P 0.180  R 0.360 

## Making the fold3 submission: decoder_full

Running predict.py on the fold3 test documents with merge decoding, to
write prediction.json and submission.zip for Codabench.


In [6]:
!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/decoder_full/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/test_text.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p2/decoder_full \
    --decoding merge


Loaded decoder NER checkpoint from step 250, val loss 0.1496
Decoding: merge
tagged 5/35 documents
tagged 10/35 documents
tagged 15/35 documents
tagged 20/35 documents
document 1305957 is empty, giving it no spans
tagged 25/35 documents
tagged 30/35 documents
tagged 35/35 documents

Wrote /content/drive/MyDrive/CS781/submissions_p2/decoder_full/prediction.json
4671 predicted spans across 35 documents
Wrote /content/drive/MyDrive/CS781/submissions_p2/decoder_full/submission.zip (ready to upload to Codabench)

First 10 spans in document 1128835:
   [165, 183, 'APP', 'SCO FABS PVT. LTD.']
   [213, 229, 'RESP', 'STATE OF HARYANA']
   [275, 281, 'CASENO', '581 OF']
   [337, 345, 'CASENO', '15544 OF']
   [365, 368, 'RESP', 'ACH']
   [375, 391, 'APP', 'COGNISED SCHOOLS']
   [414, 416, 'RESP', 'CI']
   [418, 423, 'RESP', 'Y LTD']
   [454, 470, 'RESP', 'STATE OF HARYANA']
   [521, 523, 'CASENO', '30']


## LoRA: decoder

Starting from the Phase-1 decoder checkpoint again, but freezing all of
its weights and training only small LoRA matrices (rank 8) on every
attention layer, plus the classifier head.


In [7]:
!python /content/drive/MyDrive/CS781/phase2/train.py --arch decoder --method lora \
    --learning_rate 5e-4 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p2 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/decoder_lora


Using device: cuda
Number of labels: 25
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: lora, training 0.46M of 110.0M parameters (0.42%)
524 train chunks, 527 dev chunks
131 steps per epoch, 393 steps total
step 10/393  epoch 1  loss 3.8535  lr 1.00e-04  elapsed 0.1 min
step 20/393  epoch 1  loss 3.2201  lr 2.00e-04  elapsed 0.1 min
step 30/393  epoch 1  loss 1.4925  lr 3.00e-04  elapsed 0.1 min
step 40/393  epoch 1  loss 0.6301  lr 4.00e-04  elapsed 0.2 min
step 50/393  epoch 1  loss 0.3631  lr 5.00e-04  elapsed 0.2 min
  validation loss: 0.3457
step 60/393  epoch 1  loss 0.3512  lr 4.99e-04  elapsed 0.6 min
step 70/393  epoch 1  loss 0.2026  lr 4.97e-04  elapsed 0.6 min
step 80/393  epoch 1  loss 0.2391  lr 4.92e-04  elapsed 0.7 min
step 90/393  epoch 1  loss 0.1885  lr 4.86e-04  elapsed 0.7 min
step 100/393  epoch 1  loss 0.1506  lr 4.78e-04  elapsed 0.8 min
  validation loss: 0.203

## Scoring decoder_lora on fold2 and making the fold3 submission

Checking the strict F1 on fold2 first, then writing the fold3
submission with merge decoding.


In [8]:
!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/decoder_lora/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/fold2-text.json \
    --gold_file /content/drive/MyDrive/CS781/data_p2/fold2-gold.json \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/decoder_lora/dev_check \
    --decoding merge

!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/decoder_lora/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/test_text.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p2/decoder_lora \
    --decoding merge


Loaded decoder (lora) NER checkpoint from step 393, val loss 0.1505
Decoding: merge
tagged 5/35 documents
tagged 10/35 documents
tagged 15/35 documents
tagged 20/35 documents
tagged 25/35 documents
tagged 30/35 documents
tagged 35/35 documents

Wrote /content/drive/MyDrive/CS781/ckpt_p2/decoder_lora/dev_check/prediction.json
4977 predicted spans across 35 documents

Strict macro P 0.2609  R 0.3032  F1 0.2763

  APP         P 0.030  R 0.020  F1 0.024  (gold 200, pred 133)
  RESP        P 0.048  R 0.016  F1 0.024  (gold 190, pred 63)
  A.COUNSEL   P 0.299  R 0.256  F1 0.276  (gold 125, pred 107)
  R.COUNSEL   P 0.259  R 0.361  F1 0.302  (gold 97, pred 135)
  JUDGE       P 0.160  R 0.221  F1 0.186  (gold 122, pred 169)
  WIT         P 0.000  R 0.000  F1 0.000  (gold 120, pred 13)
  AUTH        P 0.151  R 0.111  F1 0.128  (gold 468, pred 345)
  COURT       P 0.666  R 0.785  F1 0.720  (gold 558, pred 658)
  STAT        P 0.426  R 0.535  F1 0.474  (gold 735, pred 923)
  PREC        P 0.132  

## LORMA: encdec

Starting from the Phase-1 encdec checkpoint, using only its encoder
(it reads in both directions). Its weights are frozen and only the LoRMA+
matrices (rank 8) on every attention layer are trained, plus the
classifier head. LoRMA multiplies the frozen weight's output instead of
adding to it like LoRA does.


In [9]:
!python /content/drive/MyDrive/CS781/phase2/train.py --arch encdec --method lorma \
    --learning_rate 5e-4 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p2 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma


Using device: cuda
Number of labels: 25
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 68
Method: lorma, training 0.29M of 59.1M parameters (0.49%)
524 train chunks, 527 dev chunks
131 steps per epoch, 393 steps total
step 10/393  epoch 1  loss 2.9052  lr 1.00e-04  elapsed 0.0 min
step 20/393  epoch 1  loss 2.1453  lr 2.00e-04  elapsed 0.1 min
step 30/393  epoch 1  loss 0.8616  lr 3.00e-04  elapsed 0.1 min
step 40/393  epoch 1  loss 0.6272  lr 4.00e-04  elapsed 0.1 min
step 50/393  epoch 1  loss 0.5120  lr 5.00e-04  elapsed 0.1 min
  validation loss: 0.5342
step 60/393  epoch 1  loss 0.6603  lr 4.99e-04  elapsed 0.3 min
step 70/393  epoch 1  loss 0.4757  lr 4.97e-04  elapsed 0.3 min
step 80/393  epoch 1  loss 0.5945  lr 4.92e-04  elapsed 0.3 min
step 90/393  epoch 1  loss 0.5305  lr 4.86e-04  elapsed 0.3 min
step 100/393  epoch 1  loss 0.4185  lr 4.78e-04  elapsed 0.4 min
  validation loss: 0.537

## Scoring encdec_lorma on fold2 and making the fold3 submission

Checking the strict F1 on fold2 first, then writing the fold3
submission with merge decoding.


In [10]:
!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/fold2-text.json \
    --gold_file /content/drive/MyDrive/CS781/data_p2/fold2-gold.json \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma/dev_check \
    --decoding merge

!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/test_text.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p2/encdec_lorma \
    --decoding merge


Loaded encdec (lorma) NER checkpoint from step 393, val loss 0.3404
Decoding: merge
tagged 5/35 documents
tagged 10/35 documents
tagged 15/35 documents
tagged 20/35 documents
tagged 25/35 documents
tagged 30/35 documents
tagged 35/35 documents

Wrote /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma/dev_check/prediction.json
2176 predicted spans across 35 documents

Strict macro P 0.0119  R 0.0122  F1 0.0120

  APP         P 0.000  R 0.000  F1 0.000  (gold 200, pred 0)
  RESP        P 0.000  R 0.000  F1 0.000  (gold 190, pred 0)
  A.COUNSEL   P 0.000  R 0.000  F1 0.000  (gold 125, pred 0)
  R.COUNSEL   P 0.000  R 0.000  F1 0.000  (gold 97, pred 0)
  JUDGE       P 0.000  R 0.000  F1 0.000  (gold 122, pred 0)
  WIT         P 0.000  R 0.000  F1 0.000  (gold 120, pred 0)
  AUTH        P 0.000  R 0.000  F1 0.000  (gold 468, pred 0)
  COURT       P 0.000  R 0.000  F1 0.000  (gold 558, pred 0)
  STAT        P 0.000  R 0.000  F1 0.000  (gold 735, pred 0)
  PREC        P 0.002  R 0.004  F1 0.00

## Training encdec LORMA further

The 3 epoch run only got 0.012 F1 on fold2 and its loss was still going
down, so training again for 8 epochs with a higher learning rate(1e-3).


In [11]:
!python /content/drive/MyDrive/CS781/phase2/train.py --arch encdec --method lorma \
    --epochs 8 --learning_rate 1e-3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p2 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma_v2


Using device: cuda
Number of labels: 25
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 68
Method: lorma, training 0.29M of 59.1M parameters (0.49%)
524 train chunks, 527 dev chunks
131 steps per epoch, 1048 steps total
step 10/1048  epoch 1  loss 2.7778  lr 2.00e-04  elapsed 0.0 min
step 20/1048  epoch 1  loss 1.3815  lr 4.00e-04  elapsed 0.1 min
step 30/1048  epoch 1  loss 0.5409  lr 6.00e-04  elapsed 0.1 min
step 40/1048  epoch 1  loss 0.6092  lr 8.00e-04  elapsed 0.1 min
step 50/1048  epoch 1  loss 0.5151  lr 1.00e-03  elapsed 0.1 min
  validation loss: 0.5339
step 60/1048  epoch 1  loss 0.6532  lr 1.00e-03  elapsed 0.3 min
step 70/1048  epoch 1  loss 0.4670  lr 9.99e-04  elapsed 0.3 min
step 80/1048  epoch 1  loss 0.6118  lr 9.98e-04  elapsed 0.3 min
step 90/1048  epoch 1  loss 0.5272  lr 9.97e-04  elapsed 0.3 min
step 100/1048  epoch 1  loss 0.3951  lr 9.95e-04  elapsed 0.4 min
  validation 

## Scoring encdec_lorma_v2 on fold2 and making the fold3 submission

Checking the strict F1 on fold2 first, then writing the fold3
submission with merge decoding.


In [12]:
!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma_v2/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/fold2-text.json \
    --gold_file /content/drive/MyDrive/CS781/data_p2/fold2-gold.json \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma_v2/dev_check \
    --decoding merge

!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma_v2/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/test_text.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p2/encdec_lorma_v2 \
    --decoding merge


Loaded encdec (lorma) NER checkpoint from step 1048, val loss 0.2562
Decoding: merge
tagged 5/35 documents
tagged 10/35 documents
tagged 15/35 documents
tagged 20/35 documents
tagged 25/35 documents
tagged 30/35 documents
tagged 35/35 documents

Wrote /content/drive/MyDrive/CS781/ckpt_p2/encdec_lorma_v2/dev_check/prediction.json
4728 predicted spans across 35 documents

Strict macro P 0.1349  R 0.1439  F1 0.1360

  APP         P 0.000  R 0.000  F1 0.000  (gold 200, pred 1)
  RESP        P 0.000  R 0.000  F1 0.000  (gold 190, pred 12)
  A.COUNSEL   P 0.000  R 0.000  F1 0.000  (gold 125, pred 0)
  R.COUNSEL   P 0.000  R 0.000  F1 0.000  (gold 97, pred 0)
  JUDGE       P 0.000  R 0.000  F1 0.000  (gold 122, pred 6)
  WIT         P 0.000  R 0.000  F1 0.000  (gold 120, pred 0)
  AUTH        P 0.205  R 0.188  F1 0.196  (gold 468, pred 429)
  COURT       P 0.748  R 0.649  F1 0.695  (gold 558, pred 484)
  STAT        P 0.152  R 0.113  F1 0.130  (gold 735, pred 545)
  PREC        P 0.022  R 0.0

## BitFit: moe

Starting from the Phase-1 moe checkpoint. Its Linear layers were built
without biases, so a zero bias is added to each one and only those biases
are trained, plus the classifier head. Training for 6 epochs with lr 1e-3,
since BitFit trains very few weights.


In [13]:
!python /content/drive/MyDrive/CS781/phase2/train.py --arch moe --method bitfit \
    --epochs 6 --learning_rate 1e-3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p2 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/moe_bitfit


Using device: cuda
Number of labels: 25
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: bitfit, training 0.26M of 194.5M parameters (0.13%)
524 train chunks, 527 dev chunks
131 steps per epoch, 786 steps total
step 10/786  epoch 1  loss 3.2319  lr 2.00e-04  elapsed 0.0 min
step 20/786  epoch 1  loss 1.5016  lr 4.00e-04  elapsed 0.1 min
step 30/786  epoch 1  loss 0.5024  lr 6.00e-04  elapsed 0.1 min
step 40/786  epoch 1  loss 0.4569  lr 8.00e-04  elapsed 0.2 min
step 50/786  epoch 1  loss 0.2998  lr 1.00e-03  elapsed 0.2 min
  validation loss: 0.3016
step 60/786  epoch 1  loss 0.3003  lr 1.00e-03  elapsed 0.5 min
step 70/786  epoch 1  loss 0.1958  lr 9.99e-04  elapsed 0.5 min
step 80/786  epoch 1  loss 0.2374  lr 9.97e-04  elapsed 0.5 min
step 90/786  epoch 1  loss 0.1883  lr 9.94e-04  elapsed 0.6 min
step 100/786  epoch 1  loss 0.1674  lr 9.90e-04  elapsed 0.6 min
  validation loss: 0.2178


## Scoring moe_bitfit on fold2 and making the fold3 submission

Checking the strict F1 on fold2 first, then writing the fold3
submission with merge decoding.


In [14]:
!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/moe_bitfit/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/fold2-text.json \
    --gold_file /content/drive/MyDrive/CS781/data_p2/fold2-gold.json \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/moe_bitfit/dev_check \
    --decoding merge

!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/moe_bitfit/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/test_text.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p2/moe_bitfit \
    --decoding merge


Loaded moe (bitfit) NER checkpoint from step 786, val loss 0.1684
Decoding: merge
tagged 5/35 documents
tagged 10/35 documents
tagged 15/35 documents
tagged 20/35 documents
tagged 25/35 documents
tagged 30/35 documents
tagged 35/35 documents

Wrote /content/drive/MyDrive/CS781/ckpt_p2/moe_bitfit/dev_check/prediction.json
5390 predicted spans across 35 documents

Strict macro P 0.2441  R 0.2869  F1 0.2525

  APP         P 0.044  R 0.040  F1 0.042  (gold 200, pred 181)
  RESP        P 0.068  R 0.026  F1 0.038  (gold 190, pred 73)
  A.COUNSEL   P 0.211  R 0.120  F1 0.153  (gold 125, pred 71)
  R.COUNSEL   P 0.250  R 0.515  F1 0.337  (gold 97, pred 200)
  JUDGE       P 0.123  R 0.172  F1 0.143  (gold 122, pred 171)
  WIT         P 0.120  R 0.025  F1 0.041  (gold 120, pred 25)
  AUTH        P 0.178  R 0.137  F1 0.155  (gold 468, pred 360)
  COURT       P 0.699  R 0.774  F1 0.735  (gold 558, pred 618)
  STAT        P 0.315  R 0.389  F1 0.348  (gold 735, pred 907)
  PREC        P 0.097  R 0.1

## Partial fine-tuning: convmix

Starting from the Phase-1 convmix checkpoint. The embeddings and the lower
8 blocks are frozen, and only the top 4 blocks, the final norm and the
classifier head are trained.


In [15]:
!python /content/drive/MyDrive/CS781/phase2/train.py --arch convmix --method partial \
    --train_top_layers 4 --learning_rate 1e-4 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p2 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/convmix_partial


Using device: cuda
Number of labels: 25
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: partial, training 27.17M of 104.9M parameters (25.91%)
524 train chunks, 527 dev chunks
131 steps per epoch, 393 steps total
step 10/393  epoch 1  loss 3.0843  lr 2.00e-05  elapsed 0.0 min
step 20/393  epoch 1  loss 1.4102  lr 4.00e-05  elapsed 0.1 min
step 30/393  epoch 1  loss 0.4282  lr 6.00e-05  elapsed 0.1 min
step 40/393  epoch 1  loss 0.3079  lr 8.00e-05  elapsed 0.1 min
step 50/393  epoch 1  loss 0.2030  lr 1.00e-04  elapsed 0.2 min
  validation loss: 0.2378
step 60/393  epoch 1  loss 0.2351  lr 9.98e-05  elapsed 0.4 min
step 70/393  epoch 1  loss 0.1431  lr 9.93e-05  elapsed 0.5 min
step 80/393  epoch 1  loss 0.1844  lr 9.84e-05  elapsed 0.5 min
step 90/393  epoch 1  loss 0.1436  lr 9.72e-05  elapsed 0.5 min
step 100/393  epoch 1  loss 0.1266  lr 9.55e-05  elapsed 0.6 min
  validation loss: 

## Scoring convmix_partial on fold2 and making the fold3 submission

Checking the strict F1 on fold2 first, then writing the fold3
submission with merge decoding.


In [16]:
!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/convmix_partial/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/fold2-text.json \
    --gold_file /content/drive/MyDrive/CS781/data_p2/fold2-gold.json \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/convmix_partial/dev_check \
    --decoding merge

!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/convmix_partial/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/test_text.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p2/convmix_partial \
    --decoding merge


Loaded convmix (partial) NER checkpoint from step 250, val loss 0.1545
Decoding: merge
tagged 5/35 documents
tagged 10/35 documents
tagged 15/35 documents
tagged 20/35 documents
tagged 25/35 documents
tagged 30/35 documents
tagged 35/35 documents

Wrote /content/drive/MyDrive/CS781/ckpt_p2/convmix_partial/dev_check/prediction.json
5465 predicted spans across 35 documents

Strict macro P 0.2824  R 0.3219  F1 0.2930

  APP         P 0.028  R 0.025  F1 0.026  (gold 200, pred 179)
  RESP        P 0.133  R 0.053  F1 0.075  (gold 190, pred 75)
  A.COUNSEL   P 0.287  R 0.328  F1 0.306  (gold 125, pred 143)
  R.COUNSEL   P 0.305  R 0.371  F1 0.335  (gold 97, pred 118)
  JUDGE       P 0.261  R 0.402  F1 0.316  (gold 122, pred 188)
  WIT         P 0.042  R 0.008  F1 0.014  (gold 120, pred 24)
  AUTH        P 0.176  R 0.122  F1 0.144  (gold 468, pred 324)
  COURT       P 0.753  R 0.776  F1 0.764  (gold 558, pred 575)
  STAT        P 0.347  R 0.439  F1 0.388  (gold 735, pred 930)
  PREC        P 0

## Knowledge distillation: decoder_full teacher, 6-layer student

The teacher is the fine-tuned decoder_full model (12 layers). The student
is a new decoder made from only the first 6 layers of the Phase-1 decoder,
with a new NER head. The student learns from the gold labels and from the
teacher's soft predictions (alpha 0.5, temperature 2), for 5 epochs.


In [2]:
!python /content/drive/MyDrive/CS781/phase2/distill.py \
    --teacher_ckpt /content/drive/MyDrive/CS781/ckpt_p2/decoder_full/best_checkpoint.pt \
    --p1_ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p2 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/decoder_distill


Using device: cuda
Student backbone loaded, missing keys: 0, unused keys: 43
Teacher: 109.5M parameters, student: 67.1M parameters (61% of the teacher)
131 steps per epoch, 655 steps total
step 10/655  epoch 1  loss 6.7464  lr 1.00e-05  elapsed 0.1 min
step 20/655  epoch 1  loss 4.3965  lr 2.00e-05  elapsed 0.1 min
step 30/655  epoch 1  loss 1.1584  lr 3.00e-05  elapsed 0.2 min
step 40/655  epoch 1  loss 0.8391  lr 4.00e-05  elapsed 0.2 min
step 50/655  epoch 1  loss 0.4985  lr 5.00e-05  elapsed 0.3 min
  validation loss: 0.3610
step 60/655  epoch 1  loss 0.4773  lr 5.00e-05  elapsed 0.5 min
step 70/655  epoch 1  loss 0.2568  lr 4.99e-05  elapsed 0.6 min
step 80/655  epoch 1  loss 0.2862  lr 4.97e-05  elapsed 0.6 min
step 90/655  epoch 1  loss 0.1960  lr 4.95e-05  elapsed 0.7 min
step 100/655  epoch 1  loss 0.1775  lr 4.93e-05  elapsed 0.7 min
  validation loss: 0.2376
step 110/655  epoch 1  loss 0.2038  lr 4.90e-05  elapsed 0.9 min
step 120/655  epoch 1  loss 0.1844  lr 4.86e-05  elap

## Scoring decoder_distill on fold2 and making the fold3 submission

Checking the strict F1 of the 6-layer student on fold2 first, then
writing the fold3 submission with merge decoding.


In [3]:
!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/decoder_distill/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/fold2-text.json \
    --gold_file /content/drive/MyDrive/CS781/data_p2/fold2-gold.json \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p2/decoder_distill/dev_check \
    --decoding merge

!python /content/drive/MyDrive/CS781/phase2/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p2/decoder_distill/best_checkpoint.pt \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --text_file /content/drive/MyDrive/CS781/data_p2/test_text.json \
    --out_dir /content/drive/MyDrive/CS781/submissions_p2/decoder_distill \
    --decoding merge


Loaded decoder (distill) NER checkpoint from step 600, val loss 0.1648
Decoding: merge
tagged 5/35 documents
tagged 10/35 documents
tagged 15/35 documents
tagged 20/35 documents
tagged 25/35 documents
tagged 30/35 documents
tagged 35/35 documents

Wrote /content/drive/MyDrive/CS781/ckpt_p2/decoder_distill/dev_check/prediction.json
4921 predicted spans across 35 documents

Strict macro P 0.2990  R 0.3263  F1 0.3060

  APP         P 0.051  R 0.030  F1 0.038  (gold 200, pred 118)
  RESP        P 0.174  R 0.079  F1 0.109  (gold 190, pred 86)
  A.COUNSEL   P 0.317  R 0.352  F1 0.333  (gold 125, pred 139)
  R.COUNSEL   P 0.298  R 0.381  F1 0.335  (gold 97, pred 124)
  JUDGE       P 0.247  R 0.361  F1 0.293  (gold 122, pred 178)
  WIT         P 0.000  R 0.000  F1 0.000  (gold 120, pred 11)
  AUTH        P 0.220  R 0.143  F1 0.173  (gold 468, pred 305)
  COURT       P 0.742  R 0.794  F1 0.767  (gold 558, pred 597)
  STAT        P 0.510  R 0.506  F1 0.508  (gold 735, pred 730)
  PREC        P 0